# ImageGrains 2.0 Web — Pre-Processing Benchmark (`validation/preprocessing/`)

Benchmarks optional client-side image pre-processing (**CLAHE**, **bilinear upscaling**, **adaptive upscaling**, **gamma/brightness invariance sanity check**, and optional **mild denoise**) using the shipped **TypeScript** `webGrainy` pipeline (`onnxruntime-web@1.31.1`, `prepareWorkingImage(..., "auto")`, `applyCLAHE`, `runUnetCellposeInference` / `runCpsamCellposeInference`, `flowsToMasksBrowser` defaults: `flow=0.45`, `cellprob=0`, `minSize=30`, `maxAreaFraction=0.40`).

- **Why `PRESET` matters:** On 5-tile parity, `ig2_cpsam_int8_wo_fp32scale.onnx` differs from Python ORT CPU (`max_abs = 0.607 > 0.05`), so it **must** run native `ort_web_wasm` (`~14.5 s/tile` at 4 WASM threads). Running all 40 test images × all variants (`~6,500+` test tiles + train `T` selection) takes `~42+ hours`.
- **Default `PRESET = "fast_cpsam_int8_spotcheck"` (~1.5 h on Colab CPU):** Runs native WASM `cpsam_int8` on 5 stratified sites (`CT, NB2, DV_4, NZ2, APF_2`, variants `A, B, C, D` = ~368 tiles) with **real-time per-tile progress & ETAs**, and merges cleanly with the completed `unet_fp32` (`n=40`) & `cpsam_fp16` (`n=5`) benchmark already in the repository. Switch `PRESET` to `"cpsam_int8_13sites"` (~3.8 h), `"cpsam_int8_core_40"` (~12.3 h, Drive-resumable), or `"full_from_scratch"` (~42+ h) as needed.
- **Resumable Drive caching:** Per-image CSVs are flushed directly to `DRIVE_CACHE_DIR/cache` after every CP-SAM image, and `results.json` / `results_per_image.csv` checkpoint after each variant (and even in `finally:` if interrupted).


In [ ]:
# CONFIG — Preprocessing Benchmark (High-RAM CPU Colab supported; GPU not required)
REPO_URL = "https://github.com/LukeAFullard/webGrainy.git"
REPO_BRANCH = "validation/preprocessing-benchmark"
REPO_DIR = "/content/webGrainy"

HF_REPO = "inYourOwnBrowser/imagegrains-2.0-onnx"
HF_REVISION = "98e76b1374dc22cfff8193aca71c308f2b4dd3b3"

# Runtime presets (cpsam_int8 uses native WASM at ~14.5s/tile on 4 threads):
#   "fast_cpsam_int8_spotcheck" -> ~1.5h : cpsam_int8, 5 key sites (CT,NB2,DV_4,NZ2,APF_2), A/B/C/D (~368 tiles) + repo merge
#   "cpsam_int8_13sites"        -> ~3.8h : cpsam_int8, 13 sites (1/site), A/B/D (~950 tiles) + repo merge
#   "cpsam_int8_core_40"        -> ~12.3h: cpsam_int8, all 40 test imgs, A/B/D (~3,056 tiles) + repo merge (Drive-resumable)
#   "full_from_scratch"         -> ~42h+ : all 3 models & all variants from scratch (multi-session Drive resume)
#   "custom"                    -> uses CUSTOM_* settings below
PRESET = "fast_cpsam_int8_spotcheck"

CUSTOM_MODELS = ["cpsam_int8"]
CUSTOM_VARIANTS = ["A", "B", "C", "D"]
CUSTOM_MAX_TEST_IMAGES = 5
CUSTOM_STRATIFIED_TEST = True

MERGE_WITH_REPO_RESULTS = True  # Keep completed unet_fp32 (n=40) & cpsam_fp16 (n=5) from repo
TRAIN_T_MAX_IMAGES = 203        # Choose T on official_split == "train" for U-Net (if "F" is included)
TRAIN_T_MAX_IMAGES_CPSAM = 13   # Stratified train images for CP-SAM T selection (if "F" is included)
WASM_NUM_THREADS = 0            # 0 = auto: min(8, max(2, cpu_count))
BRIDGE_MAX_ABS_TOL = 0.05       # 5-tile max |WASM - PyCPU|; INT8 (~0.4-0.6) falls back to native WASM
SEED = 42

DRIVE_CACHE_DIR = "/content/drive/MyDrive/webGrainy_preprocessing_audit"
DRIVE_DATA_DIR = "/content/drive/MyDrive/imagegrains_datasets"
LOCAL_WORK_DIR = "/content/prep_bench_work"
SKIP_CACHED_ON_DRIVE = True

_PRESETS = {
    "fast_cpsam_int8_spotcheck": (["cpsam_int8"], ["A", "B", "C", "D"], 5, True),
    "cpsam_int8_13sites":        (["cpsam_int8"], ["A", "B", "D"], 13, True),
    "cpsam_int8_core_40":        (["cpsam_int8"], ["A", "B", "D"], 40, False),
    "full_from_scratch": (
        ["unet_fp32", "cpsam_int8", "cpsam_fp16"],
        ["A", "B", "C", "D", "E", "F", "G_gamma07", "G_gamma14", "G_bright_p20", "G_bright_m20", "H_bilateral", "H_median3"],
        40,
        False,
    ),
    "custom": (CUSTOM_MODELS, CUSTOM_VARIANTS, CUSTOM_MAX_TEST_IMAGES, CUSTOM_STRATIFIED_TEST),
}
MODELS, VARIANTS, MAX_TEST_IMAGES, STRATIFIED_TEST = _PRESETS[PRESET]
print(f"Preset={PRESET} | MODELS={MODELS} | VARIANTS={VARIANTS} | MAX_TEST={MAX_TEST_IMAGES} (stratified={STRATIFIED_TEST})")


In [ ]:
# Cell 1: Mount Drive, clone & build webGrainy, install Python packages, verify shipped TS runners
import os, shutil, subprocess, sys

DRIVE_MOUNTED = False
if os.path.exists("/content"):
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DRIVE_MOUNTED = os.path.isdir("/content/drive/MyDrive")
    except Exception as e:
        print("Drive mount skipped:", e)

os.makedirs(LOCAL_WORK_DIR, exist_ok=True)
if DRIVE_MOUNTED:
    os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
    os.makedirs(DRIVE_DATA_DIR, exist_ok=True)

# Install Node 22+ if needed
need_node = False
try:
    v = subprocess.check_output(["node", "-v"], text=True).strip()
    if int(v.lstrip("v").split(".")[0]) < 20:
        need_node = True
except Exception:
    need_node = True

if need_node:
    subprocess.run("curl -fsSL https://deb.nodesource.com/setup_22.x | bash - && apt-get install -y -qq nodejs", shell=True, check=True)

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", "--branch", REPO_BRANCH, "--depth", "1", REPO_URL, REPO_DIR], check=True)
subprocess.run(["npm", "ci"], cwd=REPO_DIR, check=True)
subprocess.run(["npm", "run", "build"], cwd=REPO_DIR, check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "onnxruntime>=1.19", "opencv-python-headless", "pillow", "tifffile",
    "scikit-image", "scipy", "pandas", "matplotlib", "tqdm"
], check=True)

for rel in [
    "validation/heldout/ts_cellpose_runner.mjs",
    "validation/heldout/ts_postprocess_runner.mjs",
    "validation/preprocessing/ts_preprocess_worker.mjs",
]:
    subprocess.run(["node", os.path.join(REPO_DIR, rel), "--selftest"], cwd=REPO_DIR, check=True)
eff_threads = WASM_NUM_THREADS if WASM_NUM_THREADS > 0 else min(8, max(2, os.cpu_count() or 4))
print(f"Node + TS self-tests PASSED. Drive mounted: {DRIVE_MOUNTED} | CPU cores: {os.cpu_count()} (WASM threads={eff_threads})")


In [ ]:
# Cell 2: Download & SHA-256 verify HF ONNX models + Zenodo IG2_data.zip (cached on Drive)
import hashlib, urllib.request, zipfile

MODEL_FILES = {
    "unet_fp32": ("ig2_unet_fp32.onnx", "5e692c5593ac15e614b77602e1d4f8c2f84dd939f4386b1f7e5641dbcd62053c"),
    "cpsam_int8": ("ig2_cpsam_int8_wo_fp32scale.onnx", "2cda0be5b81f1d0c0ac36998beb9f741e11833aacd089c4f5cee92488bab6f43"),
    "cpsam_fp16": ("ig2_cpsam_fp16.onnx", "944a114a0cb1a3a215c11affefe7211caac6da2b74895c496d5a7b5a23b6230d"),
}

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

MODELS_DIR = os.path.join(LOCAL_WORK_DIR, "models")
os.makedirs(MODELS_DIR, exist_ok=True)
for mkey in MODELS:
    fname, expected_sha = MODEL_FILES[mkey]
    dst = os.path.join(MODELS_DIR, fname)
    drive_src = os.path.join(DRIVE_DATA_DIR, "hf_models", HF_REVISION, fname) if DRIVE_MOUNTED else ""
    if DRIVE_MOUNTED and os.path.exists(drive_src) and sha256_file(drive_src) == expected_sha:
        shutil.copy2(drive_src, dst)
    if not (os.path.exists(dst) and sha256_file(dst) == expected_sha):
        url = f"https://huggingface.co/{HF_REPO}/resolve/{HF_REVISION}/{fname}"
        print("Downloading", url)
        urllib.request.urlretrieve(url, dst)
        assert sha256_file(dst) == expected_sha, f"SHA mismatch for {fname}"
        if DRIVE_MOUNTED:
            os.makedirs(os.path.dirname(drive_src), exist_ok=True)
            shutil.copy2(dst, drive_src)
    print(f"Verified {fname} ({os.path.getsize(dst)/1e6:.1f} MB)")

IG2_ZIP_URL = "https://zenodo.org/records/17866827/files/IG2_data.zip?download=1"
DATA_DIR = os.path.join(LOCAL_WORK_DIR, "ig2_data")
if not os.path.exists(os.path.join(DATA_DIR, "CT")):
    zip_path = os.path.join(DRIVE_DATA_DIR, "IG2_data.zip") if (DRIVE_MOUNTED and os.path.exists(os.path.join(DRIVE_DATA_DIR, "IG2_data.zip"))) else os.path.join(LOCAL_WORK_DIR, "IG2_data.zip")
    if not os.path.exists(zip_path):
        print("Downloading IG2_data.zip from Zenodo...")
        urllib.request.urlretrieve(IG2_ZIP_URL, zip_path)
    os.makedirs(DATA_DIR, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(DATA_DIR)
    nested = os.path.join(DATA_DIR, "IG2_data")
    if os.path.isdir(nested):
        for name in os.listdir(nested):
            shutil.move(os.path.join(nested, name), os.path.join(DATA_DIR, name))
        os.rmdir(nested)
assert os.path.isdir(os.path.join(DATA_DIR, "CT", "test")), f"Missing extracted test folder under {DATA_DIR}"
print("IG2_data ready at:", DATA_DIR)


In [ ]:
# Cell 3: Run preprocessing benchmark with live unbuffered tile/ETA streaming & Drive checkpointing
CACHE_DIR = os.path.join(DRIVE_CACHE_DIR, "cache") if (DRIVE_MOUNTED and SKIP_CACHED_ON_DRIVE) else os.path.join(LOCAL_WORK_DIR, "cache")
OUT_DIR = os.path.join(LOCAL_WORK_DIR, "out")
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

cmd = [
    sys.executable,
    "-u",
    os.path.join(REPO_DIR, "validation/preprocessing/run_benchmark.py"),
    "--data-dir", DATA_DIR,
    "--models-dir", MODELS_DIR,
    "--out-dir", OUT_DIR,
    "--cache-dir", CACHE_DIR,
    "--models", *MODELS,
    "--variants", *VARIANTS,
    "--max-test", str(MAX_TEST_IMAGES),
    "--train-t-max", str(TRAIN_T_MAX_IMAGES),
    "--train-t-max-cpsam", str(TRAIN_T_MAX_IMAGES_CPSAM),
    "--wasm-threads", str(WASM_NUM_THREADS),
    "--bridge-tol", str(BRIDGE_MAX_ABS_TOL),
]
if STRATIFIED_TEST:
    cmd.append("--stratified-test")
repo_results = os.path.join(REPO_DIR, "validation/preprocessing/results.json")
if MERGE_WITH_REPO_RESULTS and os.path.exists(repo_results):
    cmd.extend(["--merge-existing", repo_results])

print("Running:", " ".join(cmd), flush=True)
env = {**os.environ, "PYTHONUNBUFFERED": "1"}
proc = subprocess.Popen(
    cmd,
    cwd=REPO_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)
rc = None
try:
    assert proc.stdout is not None
    for line in iter(proc.stdout.readline, ""):
        sys.stdout.write(line)
        sys.stdout.flush()
    rc = proc.wait()
finally:
    if proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=5)
        except Exception:
            proc.kill()
    if DRIVE_MOUNTED:
        for fname in ("results.json", "results_per_image.csv"):
            src = os.path.join(OUT_DIR, fname)
            if os.path.exists(src):
                shutil.copy2(src, os.path.join(DRIVE_CACHE_DIR, fname))
        print("\nSynced available results to", DRIVE_CACHE_DIR, flush=True)

if rc != 0:
    raise subprocess.CalledProcessError(rc if rc is not None else 1, cmd)


In [ ]:
# Cell 4: Summary tables, decision rule PASS/FAIL check, & charts
import json
import matplotlib.pyplot as plt
import pandas as pd

res_dir = OUT_DIR if os.path.exists(os.path.join(OUT_DIR, "results.json")) else DRIVE_CACHE_DIR
results = json.load(open(os.path.join(res_dir, "results.json")))
df = pd.read_csv(os.path.join(res_dir, "results_per_image.csv"))

for mkey, mdata in results["models"].items():
    print(f"\n==================== {mkey} ====================")
    if mkey in results.get("adaptive_threshold_train", {}):
        ch = results["adaptive_threshold_train"][mkey]["chosen"]
        print("Train-selected F rule: upscale=%s when working_med_b_px < %.1f px (train n=%d, trig=%d)" % (
            ch["upscale_variant"], ch["T_px"], ch["n_train"], ch["n_triggered"]))
    base_v = mdata["variants"].get("A")
    if base_v:
        base_pq = base_v["pq_vs_gt"]["mean"]
        base_d50 = base_v["vs_gt"]["D50"]["median_ape_pct"]
        base_r10 = base_v["size_class_recall"]["lt10"]["pooled_recall"]
        print("A (n=%d)     | PQ=%.4f | F1=%.4f | MedD50=%.2f%% | recall<10px=%.3f | estWASM=%.1fs/img (%.1fs total)" % (
            base_v["n_images"], base_pq, base_v["f1_50_vs_gt"]["mean"], base_d50, base_r10,
            base_v.get("est_wasm_wall_s_mean_per_img", float("nan")), base_v.get("est_wasm_wall_s_total", float("nan"))))

    for group_name in ("paired_vs_A", "subset_paired_vs_A"):
        if not mdata.get(group_name):
            continue
        print(f"--- {group_name} ---")
        for var, p in mdata[group_name].items():
            v = mdata["variants"][var]
            dr = p["decision_rule"]
            r10 = v["size_class_recall"]["lt10"]["pooled_recall"]
            est_img = v.get("est_wasm_wall_s_mean_per_img", float("nan"))
            est_tot = v.get("est_wasm_wall_s_total", float("nan"))
            pq_m = v["pq_vs_gt"]["mean"]
            d50_m = v["vs_gt"]["D50"]["median_ape_pct"]
            print(
                "  %-12s | PQ=%.4f (d=%+.4f [%+.4f,%+.4f]) | MedD50=%.2f%% (d=%+.2fpp [%+.2f,%+.2f]) | "
                "r<10=%.3f | estWASM=%.1fs/img (%.1fs) | worst=%s(%+.2fpp) | PASS_DEF=%s" % (
                    var, pq_m, p["mean_dPQ"], p["dPQ_ci95"][0], p["dPQ_ci95"][1],
                    d50_m, p["mean_dAPE_pp"], p["dAPE_ci95_pp"][0], p["dAPE_ci95_pp"][1],
                    r10, est_img, est_tot, p["worst_site"], p["worst_site_dMed_d50_ape_pp"], dr["pass_default"]
                )
            )

# Plot PQ vs small-grain recall for each model present
for mkey, mdata in results["models"].items():
    vars_plot = [v for v in ("A", "B", "C", "D", "E", "F") if v in mdata["variants"]]
    if len(vars_plot) < 2:
        continue
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    n_plot = mdata["variants"][vars_plot[0]]["n_images"]
    pqs = [mdata["variants"][v]["pq_vs_gt"]["mean"] for v in vars_plot]
    r10s = [mdata["variants"][v]["size_class_recall"]["lt10"]["pooled_recall"] for v in vars_plot]
    axes[0].bar(vars_plot, pqs, color="#2b6cb0")
    axes[0].set_title(f"{mkey} (n={n_plot}): Mean PQ vs GT")
    axes[0].set_ylabel("PQ")
    axes[1].bar(vars_plot, r10s, color="#2f855a")
    axes[1].set_title(f"{mkey} (n={n_plot}): Small-Grain Recall (<10 px)")
    axes[1].set_ylabel("Pooled Recall (IoU > 0.5)")
    plt.tight_layout()
    chart_p = os.path.join(res_dir, f"summary_{mkey}.png")
    plt.savefig(chart_p, dpi=150)
    plt.show()
